In [ ]:
import uuid, time
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.writer import DataFrameWriter

In [ ]:
layer = "gold"
dataset = "fact_sales"

# Load Spark Session
spark = get_spark_session()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

# Create logical variables
run_id = str(uuid.uuid4())

In [ ]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")

start = time.time()
log.info(f"[GOLD][START] run_id={run_id}")

try:

    writer = DataFrameWriter(spark)
    
    fact_sales = spark.sql("""
        WITH total_order_value AS(
            SELECT
                oi.order_id,
                SUM(oi.price + oi.freight_value) AS total_order_value
            FROM olist.silver.order_items oi
            GROUP BY oi.order_id
        )
        SELECT
            o.order_id,
            o.customer_id,
            o.order_status,
            tov.total_order_value,
            c.customer_state,
            datediff(o.order_delivered_customer_date, o.order_purchase_timestamp) AS delivery_time_days
        FROM olist.silver.orders as o
        LEFT JOIN total_order_value as tov ON o.order_id = tov.order_id
        LEFT JOIN olist.silver.customers as c ON o.customer_id = c.customer_id
    """)
    
    writer.write_delta_batch(fact_sales, "olist", "gold", "fact_sales", mode="overwrite")
    duration_s = round(time.time() - start, 3)
    log.info(f"[GOLD][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[GOLD][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise